First run the evaluation with

```
inspect eval evaluate.py --model openai/gpt-5-mini -T raw_data_file=path/to/raw/trial/data.parquet
```

Then add both the cohort file path and the eval logfile to `.env`:

```
COHORT_FILE_PATH=path/to/raw/trial/data.parquet
EVAL_LOGFILE=path/to/eval/result.eval
```

Then run this notebook for analysis.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

COHORT_FILE_PATH = os.environ["COHORT_FILE_PATH"]
EVAL_LOGFILE = os.environ["EVAL_LOGFILE"]
RNG_SEED = 42

In [ ]:
from preprocess import Preprocessor

preprocessor = Preprocessor(COHORT_FILE_PATH)
preprocessor.apply_filters_for_eval()
print(preprocessor.summary())

In [ ]:
from postprocess import Postprocessor
from typing import Literal
from utils import ObservedEvalPerformance

eval_results: dict[
    Literal["addition", "removal", "tf_change"], ObservedEvalPerformance
] = {
    "tf_change": ObservedEvalPerformance(tp=22, tn=64, fp=14, fn=0),
    "addition": ObservedEvalPerformance(tp=18, tn=79, fp=2, fn=1),
    "removal": ObservedEvalPerformance(tp=18, tn=77, fp=5, fn=0),
}

postprocessor = Postprocessor(
    trials_data_path=COHORT_FILE_PATH,
    model_eval_log=EVAL_LOGFILE,
    cache_dir="cache",
    eval_results=eval_results,
    # refresh_cache=True,
    n_simulations=100,
    per_rate_samples=100,
    random_state=RNG_SEED,
)

In [ ]:
postprocessor.cohort_summary_table()

In [ ]:
Postprocessor.estimate_change_prevalences(postprocessor.df)